# TCC — Pipeline completo no Google Colab

Notebook único para GPU, Drive, GitHub, Roboflow, SAM, clustering, Streamlit e experimentos YOLO26 Detection/Segmentation em Base, LoRA e Full fine-tune.

**Regra do ambiente:** não atualizar `pip`, `setuptools` ou `wheel`; não instalar o `ultralytics` oficial separado. O notebook instala diretamente uma cópia não-editável do Tencent/YOLO-Master e valida o import antes de continuar.

Antes de executar: selecione GPU no Colab e configure em Secrets `GITHUB_TOKEN`, `ROBOFLOW_API_KEY` e, opcionalmente, `NGROK_AUTHTOKEN`.


## 1. Verificar GPU sem importar PyTorch/Ultralytics


In [ ]:
import os, sys, subprocess
print('Python:', sys.version.split()[0])
print('Interpretador:', sys.executable)
subprocess.run(['nvidia-smi'], check=False)


## 2. Montar Google Drive


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
drive.mount('/content/drive')
RUNTIME = Path('/content/drive/MyDrive/TCC_Code_Test_runtime')
for name in ['data', 'workspace', 'exports', 'weights']:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)
print('Persistência:', RUNTIME)


## 3. Clonar ou atualizar o repositório privado


In [ ]:
REPO = Path('/content/TCC_Code_Test')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('GITHUB_TOKEN não encontrado em Colab > Secrets.')
netrc = Path.home() / '.netrc'
netrc.write_text(f'machine github.com\nlogin x-access-token\npassword {token}\n', encoding='utf-8')
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/ferreis/TCC_Code_Test.git', str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)
os.chdir(REPO)
print('Commit TCC:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 4. Instalar e VALIDAR o ambiente YOLO em uma única célula

Esta célula instala as dependências sem substituir o PyTorch do Colab, instala o YOLO-Master de forma **não-editável** e só termina se um novo processo Python conseguir importar `ultralytics` e `ultralytics.utils.lora`. Se houver erro, pare aqui e leia a saída desta célula.


In [ ]:
os.chdir(REPO)
print('1/3 Instalando dependências do projeto...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '-r', 'requirements-colab.txt'
], check=True)

print('2/3 Instalando Tencent/YOLO-Master no MESMO interpretador...')
subprocess.run([
    sys.executable, 'scripts/setup_yolo_master.py',
    '--dir', '/content/YOLO-Master'
], check=True)

print('3/3 Verificação externa independente...')
check_code = r'''
import importlib.util, pathlib, sys
import ultralytics
print('Python:', sys.executable)
print('Ultralytics/YOLO-Master:', ultralytics.__version__)
print('Caminho:', pathlib.Path(ultralytics.__file__).resolve())
ok = importlib.util.find_spec('ultralytics.utils.lora') is not None
print('LoRA disponível:', ok)
if not ok:
    raise SystemExit('ERRO: módulo ultralytics.utils.lora não encontrado')
'''
subprocess.run([sys.executable, '-c', check_code], check=True)
print('AMBIENTE YOLO VALIDADO COM SUCESSO.')


## 5. Validar o mesmo ambiente dentro do kernel do notebook


In [ ]:
import importlib, importlib.util, pathlib
importlib.invalidate_caches()
spec = importlib.util.find_spec('ultralytics')
if spec is None:
    raise RuntimeError('ultralytics não está visível para este kernel, embora a etapa 4 devesse ter validado o mesmo sys.executable.')
import torch
import ultralytics
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM:', round(p.total_memory / 1024**3, 2), 'GB')
print('Ultralytics/YOLO-Master:', ultralytics.__version__)
print('Caminho:', pathlib.Path(ultralytics.__file__).resolve())
lora_ok = importlib.util.find_spec('ultralytics.utils.lora') is not None
print('LoRA disponível:', lora_ok)
assert lora_ok, 'Backend LoRA não disponível.'


## 6. Conectar data/workspace/exports/weights ao Drive


In [ ]:
import shutil
for name in ['data', 'workspace', 'exports', 'weights']:
    target = RUNTIME / name
    link = REPO / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            for p in link.iterdir():
                dst = target / p.name
                if not dst.exists():
                    shutil.copytree(p, dst) if p.is_dir() else shutil.copy2(p, dst)
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)
print('Drive conectado ao projeto.')


## 7. Configurar Roboflow e baixar dataset de detecção


In [ ]:
rf_key = userdata.get('ROBOFLOW_API_KEY')
if not rf_key:
    raise RuntimeError('ROBOFLOW_API_KEY não encontrado em Colab > Secrets.')
os.environ['ROBOFLOW_API_KEY'] = rf_key
os.environ['ROBOFLOW_WORKSPACE'] = 'rafael-fernando-dos-reis-mecabo'
os.environ['ROBOFLOW_PROJECT'] = 'detecao-de-cetaceos-afrzs-iqey9'
os.environ['ROBOFLOW_VERSION'] = '1'
images_dir = Path('data/roboflow/train/images')
if images_dir.exists() and any(images_dir.iterdir()):
    print('Dataset já existe:', images_dir)
else:
    subprocess.run([sys.executable, 'scripts/download_roboflow.py', '--format', 'yolo26', '--output', 'data/roboflow', '--overwrite'], check=True)


## 8. Baixar SAM


In [ ]:
sam = Path('weights/sam_vit_b_01ec64.pth')
if sam.exists():
    print('SAM já existe:', sam)
else:
    subprocess.run([sys.executable, 'scripts/download_sam.py'], check=True)


## 9. Clustering das bounding boxes já revisadas — opcional


In [ ]:
from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering
project_dir = Path('workspace/bbox_clusters_colab')
if (project_dir / 'catalog.json').exists():
    print('Projeto já existe:', project_dir)
else:
    cfg = BoxClusteringConfig(context_scale=1.8, min_crop_size_px=64, min_cluster_size=6, embedding_batch_size=8, dino_cpu_fallback_on_oom=True)
    result = run_box_clustering('data/roboflow/train', project_dir, cfg, overwrite=False, progress_cb=lambda d,t,m: print(f'[{d}/{t}] {m}') if d % 100 == 0 or d == t else None)
    print('Imagens:', len(result['images']), 'Objetos:', len(result['objects']))


## 10. Abrir Streamlit — opcional


In [ ]:
from pyngrok import ngrok
import time
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
if not ngrok_token:
    print('NGROK_AUTHTOKEN não configurado; etapa opcional pulada.')
else:
    ngrok.kill()
    ngrok.set_auth_token(ngrok_token)
    env = os.environ.copy()
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    streamlit_process = subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(3)
    public_url = ngrok.connect(8501, bind_tls=True).public_url
    print('Abra a interface:', public_url)


## 11. Encerrar Streamlit antes dos treinamentos


In [ ]:
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
try:
    ngrok.kill()
except Exception:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()
subprocess.run(['nvidia-smi'], check=False)


# Treinamento YOLO26
Detection e Segmentation, cada um em Base, LoRA e Full fine-tune. A primeira rodada usa YOLO26n.


## 12. Listar e validar a matriz experimental


In [ ]:
subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'all', '--list'], check=True)
subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'experiments', '--dry-run'], check=True)


## 13. Smoke test YOLO26n Detection — 3 épocas


In [ ]:
subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'smoke_tests', '--only', 'detect_yolo26n_smoke'], check=True)


## 14. Detection: Base + LoRA + Full fine-tune


In [ ]:
subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'detect_base', 'detect_lora', 'detect_full'], check=True)


## 15. Segmentation: smoke + Base + LoRA + Full fine-tune
Só executa quando `data/segmentation/data.yaml` existir.


In [ ]:
seg = Path('data/segmentation/data.yaml')
print('Dataset de segmentação pronto:', seg.exists(), '-', seg)
if seg.exists():
    subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'smoke_tests', '--only', 'segment_yolo26n_smoke'], check=True)
    subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'segment_base', 'segment_lora', 'segment_full'], check=True)
else:
    print('Etapa pulada: gere primeiro data/segmentation/data.yaml.')


## 16. Comparar parâmetros e métricas


In [ ]:
import pandas as pd
summary = Path('workspace/training_runs/experiment_summary.csv')
if summary.exists():
    display(pd.read_csv(summary))
else:
    print('A tabela aparecerá após o primeiro experimento.')


## 17. Retomar após desconexão


In [ ]:
# subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'detect_base', '--resume'], check=True)
# subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'detect_lora', '--resume'], check=True)
# subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'detect_full', '--resume'], check=True)
# subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'segment_base', '--resume'], check=True)
# subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'segment_lora', '--resume'], check=True)
# subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'segment_full', '--resume'], check=True)


## Persistência
Dados, checkpoints e métricas ficam em `MyDrive/TCC_Code_Test_runtime/`.
